### INITIAL TURBULENCE-STATE STATISTICS AFTER MITgcm SPIN-UP
## Metrics:
###   - mesoscale EKE
###   - Re_eddy
###   - MLD from GLORYS
###   - bulk Richardson number


In [3]:
from pathlib import Path
import json
import numpy as np
import xarray as xr

In [5]:
# ============================================================
# SETTINGS
# ============================================================

NB_DIR = Path.cwd()

case_name = "run_domain_characterization"          # edit
RUN_TITLE_INFO = "run_domain_characterization"     # edit
MITGCM_FILE = (NB_DIR / f"../data/input/{case_name}.nc").resolve()
ds  = xr.open_dataset(MITGCM_FILE)
ds

<xarray.Dataset> Size: 4MB
Dimensions:      (Zmd000002: 2, T: 1, Y: 512, Xp1: 513, Yp1: 513, X: 512)
Coordinates:
  * T            (T) float64 8B 0.0
  * Y            (Y) float64 4kB 250.0 750.0 1.25e+03 ... 2.552e+05 2.558e+05
  * Xp1          (Xp1) float64 4kB 0.0 500.0 1e+03 ... 2.555e+05 2.56e+05
  * Yp1          (Yp1) float64 4kB 0.0 500.0 1e+03 ... 2.555e+05 2.56e+05
  * X            (X) float64 4kB 250.0 750.0 1.25e+03 ... 2.552e+05 2.558e+05
Dimensions without coordinates: Zmd000002
Data variables:
    diag_levels  (Zmd000002) float64 16B ...
    iter         (T) int32 4B ...
    UVEL         (T, Zmd000002, Y, Xp1) float32 2MB ...
    VVEL         (T, Zmd000002, Yp1, X) float32 2MB ...
Attributes: (12/18)
    MITgcm_version:  checkpoint69l
    build_user:      jgortemaker
    build_host:      cmp060
    build_date:      Fri May 22 09:34:05 CEST 2026
    MITgcm_URL:      http://mitgcm.org
    MITgcm_tag_id:   
    ...              ...
    nSy:             1
    nPx:             8
    nPy:             8
    Nx:              512
    Ny:              512
    Nr:              84

In [4]:
# ============================================================
# SETTINGS
# ============================================================

NB_DIR = Path.cwd()

case_name = "run_domain_characterization"          # edit
RUN_TITLE_INFO = "run_domain_characterization"     # edit

MITGCM_FILE = (NB_DIR / f"../data/input/{case_name}.nc").resolve()

# Optional GLORYS MLD file. If you already extracted a scalar MLD,
# set GLORYS_MLD_FILE = None and use MLD_VALUE_M below

glorys_file_name = "GPGP_aug2020_3D_21.5_24.5_141.6_138.4"
# GLORYS_MLD_FILE = NB_DIR / f"../../OGCM/data/input/{glorys_file_name}.nc"

GLORYS_MLD_FILE = None 
GLORYS_MLD_VAR = "mlotst"      # common CMEMS/GLORYS name; edit if needed

# If no GLORYS MLD file is used, provide scalar MLD here [m]
MLD_VALUE_M = 20             # e.g. 35.0

# Time selection
SPINUP_DAYS = 2.0
SECONDS_PER_DAY = 86400.0

# Eddy-scale Reynolds number settings
L_EDDY_M = None                # if None, uses 0.25 * domain width
NU_EFF = 50.0                  # [m2/s] effective horizontal viscosity; EDIT to your setup

# Linear EOS constants
RHO0 = 1035.0                  # [kg/m3]
ALPHA_T = 2.0e-4               # [1/K]
BETA_S = 7.4e-4                # [1/psu]
G = 9.81

# Minimum shear to avoid infinite Richardson numbers
MIN_DELTA_U2 = 1e-10

# Output
OUT_DIR = (NB_DIR / "../results/domain_attrs").resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_ATTRS_FILE = OUT_DIR / f"{RUN_TITLE_INFO}_spinup_stats_attrs.json"


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def find_dim(dims, candidates):
    for c in candidates:
        if c in dims:
            return c
    for d in dims:
        for c in candidates:
            if c.lower() in d.lower():
                return d
    raise KeyError(f"Could not find dimension from candidates {candidates}. Available: {list(dims)}")


def find_var(ds, candidates):
    for c in candidates:
        if c in ds.variables:
            return c
    for v in ds.variables:
        for c in candidates:
            if c.lower() == v.lower():
                return v
    raise KeyError(f"Could not find variable from candidates {candidates}. Available: {list(ds.variables)}")


def nearest_time_index(ds, time_dim, spinup_days):
    """Select nearest model output to requested spin-up time."""
    if time_dim in ds.coords:
        t = ds[time_dim].values.astype(float)
        target_seconds = spinup_days * SECONDS_PER_DAY

        # MITgcm often stores T in seconds.
        # If values are small indices, this still falls back reasonably.
        if np.nanmax(t) > 1000:
            return int(np.nanargmin(np.abs(t - target_seconds)))
        else:
            # Assume regular 0.5-day style output if T is index-like.
            return int(round(spinup_days / 0.5))

    return int(round(spinup_days / 0.5))


def center_u(u):
    """
    MITgcm UVEL is typically on Xp1.
    Input shape: (..., Y, Xp1)
    Output shape: (..., Y, X)
    """
    return 0.5 * (u[..., :, :-1] + u[..., :, 1:])


def center_v(v):
    """
    MITgcm VVEL is typically on Yp1.
    Input shape: (..., Yp1, X)
    Output shape: (..., Y, X)
    """
    return 0.5 * (v[..., :-1, :] + v[..., 1:, :])


def crop_to_common(*arrays):
    """Crop arrays to common trailing horizontal shape."""
    ny = min(a.shape[-2] for a in arrays)
    nx = min(a.shape[-1] for a in arrays)
    return tuple(a[..., :ny, :nx] for a in arrays)


def area_mean(q, lat=None):
    """
    Area mean. If lat is provided, uses cos(lat) weighting.
    Otherwise simple nanmean.
    """
    q = np.asarray(q, dtype=float)

    if lat is None:
        return float(np.nanmean(q))

    lat = np.asarray(lat, dtype=float)
    w = np.cos(np.deg2rad(lat))

    if w.ndim == 1:
        w = w[:, None]

    q, w = crop_to_common(q, np.broadcast_to(w, q.shape))
    return float(np.nansum(q * w) / np.nansum(w * np.isfinite(q)))


def get_vertical_coordinate(ds, z_dim):
    """
    Return positive-down depth vector [m].
    Tries actual coordinate first; otherwise uses integer levels.
    """
    if z_dim in ds.coords:
        z = np.asarray(ds[z_dim].values, dtype=float)
    elif "diag_levels" in ds.variables:
        z = np.asarray(ds["diag_levels"].values, dtype=float)
    else:
        raise KeyError(
            "Could not find vertical coordinate. Add a depth coordinate or diag_levels."
        )

    z = np.squeeze(z)

    # Ensure positive downward.
    if np.nanmean(z) < 0:
        z = -z

    return z


def interp_profile_to_depth(field_zyx, z_m, target_depth_m):
    """
    Interpolate 3D field(z,y,x) to scalar target depth.
    Returns 2D field(y,x).
    """
    field_zyx = np.asarray(field_zyx, dtype=float)
    z_m = np.asarray(z_m, dtype=float)

    if target_depth_m <= z_m[0]:
        return field_zyx[0, :, :]

    if target_depth_m >= z_m[-1]:
        return field_zyx[-1, :, :]

    k2 = int(np.searchsorted(z_m, target_depth_m))
    k1 = k2 - 1

    z1 = z_m[k1]
    z2 = z_m[k2]

    w = (target_depth_m - z1) / (z2 - z1)

    return (1.0 - w) * field_zyx[k1, :, :] + w * field_zyx[k2, :, :]


def density_linear(theta, salt):
    """
    Density using linear EOS.
    Reference T/S cancel for density differences, so they are not needed here.
    """
    return RHO0 * (1.0 - ALPHA_T * theta + BETA_S * salt)


def load_mld_from_glorys(glorys_file, mld_var):
    """
    Load MLD field from GLORYS file and return mean/median/std.
    Assumes file is already cropped to the candidate domain, or contains only
    the relevant domain.
    """
    dsg = xr.open_dataset(glorys_file)

    if mld_var not in dsg.variables:
        raise KeyError(
            f"MLD variable '{mld_var}' not found. Available variables: {list(dsg.variables)}"
        )

    mld = dsg[mld_var]

    # If time exists, take first time. Edit here if needed.
    for d in mld.dims:
        if d.lower() in ["time", "t"]:
            mld = mld.isel({d: 0})

    mld_vals = np.asarray(mld.values, dtype=float)
    mld_vals = mld_vals[np.isfinite(mld_vals)]

    return {
        "mld_mean_m": float(np.nanmean(mld_vals)),
        "mld_median_m": float(np.nanmedian(mld_vals)),
        "mld_std_m": float(np.nanstd(mld_vals)),
        "mld_source": str(glorys_file),
    }


# ============================================================
# OPEN MITgcm DATA
# ============================================================

print(f"Opening MITgcm file:\n{MITGCM_FILE}")
ds = xr.open_dataset(MITGCM_FILE)

u_name = find_var(ds, ["UVEL", "uvel", "u"])
v_name = find_var(ds, ["VVEL", "vvel", "v"])
t_name = find_var(ds, ["THETA", "theta", "TEMP", "temperature", "T"])
s_name = find_var(ds, ["SALT", "salt", "salinity", "S"])

time_dim = find_dim(ds.dims, ["T", "time", "iter"])
z_dim = find_dim(ds[u_name].dims, ["Zmd", "Z", "depth", "k"])

it = nearest_time_index(ds, time_dim, SPINUP_DAYS)
time_days = SPINUP_DAYS

if time_dim in ds.coords:
    t_val = float(ds[time_dim].isel({time_dim: it}).values)
    if t_val > 1000:
        time_days = t_val / SECONDS_PER_DAY
    else:
        time_days = it * 0.5

print(f"Selected time index: {it}")
print(f"Approx. selected time: {time_days:.2f} days")

z_m = get_vertical_coordinate(ds, z_dim)

print(f"Vertical coordinate found with {len(z_m)} levels.")
print(f"Maximum available depth: {np.nanmax(z_m):.1f} m")


# ============================================================
# EXTRACT FIELDS AT SELECTED TIME
# ============================================================

u_3d_raw = ds[u_name].isel({time_dim: it}).values
v_3d_raw = ds[v_name].isel({time_dim: it}).values
theta_3d = ds[t_name].isel({time_dim: it}).values
salt_3d = ds[s_name].isel({time_dim: it}).values

# Center C-grid velocities to tracer-like grid
u_3d = center_u(u_3d_raw)
v_3d = center_v(v_3d_raw)

u_3d, v_3d, theta_3d, salt_3d = crop_to_common(
    u_3d, v_3d, theta_3d, salt_3d
)


# ============================================================
# 1. MESOSCALE EKE
# ============================================================

u_surf = u_3d[0, :, :]
v_surf = v_3d[0, :, :]

u_prime = u_surf - np.nanmean(u_surf)
v_prime = v_surf - np.nanmean(v_surf)

eke_field = 0.5 * (u_prime**2 + v_prime**2)

eke_mean = float(np.nanmean(eke_field))
eke_median = float(np.nanmedian(eke_field))
eke_std = float(np.nanstd(eke_field))

u_eddy = float(np.sqrt(2.0 * eke_mean))

print(f"EKE mean: {eke_mean:.4e} m2/s2")
print(f"U_eddy : {u_eddy:.4e} m/s")


# ============================================================
# 2. EDDY-SCALE REYNOLDS NUMBER
# ============================================================

# Estimate domain width from X coordinate if available.
if "X" in ds.coords:
    x = np.asarray(ds["X"].values, dtype=float)
    domain_width_m = float(np.nanmax(x) - np.nanmin(x))
elif "Xp1" in ds.coords:
    x = np.asarray(ds["Xp1"].values, dtype=float)
    domain_width_m = float(np.nanmax(x) - np.nanmin(x))
else:
    domain_width_m = float(u_surf.shape[-1])  # fallback: grid count, not ideal
    print("Warning: no X coordinate found. L_EDDY_M should be set manually.")

if L_EDDY_M is None:
    L_eddy_m = 0.25 * domain_width_m
else:
    L_eddy_m = float(L_EDDY_M)

re_eddy = float(u_eddy * L_eddy_m / NU_EFF)

print(f"L_eddy : {L_eddy_m:.2f} m")
print(f"nu_eff : {NU_EFF:.4e} m2/s")
print(f"Re_eddy: {re_eddy:.4e}")


# ============================================================
# 3. MLD FROM GLORYS
# ============================================================

if GLORYS_MLD_FILE is not None:
    GLORYS_MLD_FILE = Path(GLORYS_MLD_FILE).resolve()
    mld_stats = load_mld_from_glorys(GLORYS_MLD_FILE, GLORYS_MLD_VAR)
    mld_for_ri = mld_stats["mld_median_m"]
else:
    if MLD_VALUE_M is None:
        raise ValueError(
            "Provide either GLORYS_MLD_FILE or MLD_VALUE_M."
        )
    mld_for_ri = float(MLD_VALUE_M)
    mld_stats = {
        "mld_mean_m": float(MLD_VALUE_M),
        "mld_median_m": float(MLD_VALUE_M),
        "mld_std_m": np.nan,
        "mld_source": "scalar_user_input",
    }

print(f"MLD used for Ri_b: {mld_for_ri:.2f} m")


# ============================================================
# 4. BULK RICHARDSON NUMBER
# ============================================================

rho_3d = density_linear(theta_3d, salt_3d)

rho_surf = rho_3d[0, :, :]
u_surf = u_3d[0, :, :]
v_surf = v_3d[0, :, :]

rho_h = interp_profile_to_depth(rho_3d, z_m, mld_for_ri)
u_h = interp_profile_to_depth(u_3d, z_m, mld_for_ri)
v_h = interp_profile_to_depth(v_3d, z_m, mld_for_ri)

rho_surf, rho_h, u_surf, v_surf, u_h, v_h = crop_to_common(
    rho_surf, rho_h, u_surf, v_surf, u_h, v_h
)

delta_rho = rho_h - rho_surf
delta_u2 = (u_surf - u_h) ** 2 + (v_surf - v_h) ** 2

delta_u2_safe = np.where(delta_u2 < MIN_DELTA_U2, np.nan, delta_u2)

rib_field = G * delta_rho * mld_for_ri / (RHO0 * delta_u2_safe)

# Negative Ri can occur if locally unstable / noisy.
rib_median = float(np.nanmedian(rib_field))
rib_mean = float(np.nanmean(rib_field))
rib_p25 = float(np.nanpercentile(rib_field, 25))
rib_p75 = float(np.nanpercentile(rib_field, 75))

print(f"Ri_b median: {rib_median:.4e}")
print(f"Ri_b mean  : {rib_mean:.4e}")


# ============================================================
# STORE ATTRS
# ============================================================

attrs = {
    "case_name": case_name,
    "run_title_info": RUN_TITLE_INFO,
    "mitgcm_file": str(MITGCM_FILE),
    "selected_time_index": int(it),
    "selected_time_days": float(time_days),

    "statistics_description": {
        "EKE": "Domain-mean surface eddy kinetic energy after removing domain-mean velocity.",
        "Re_eddy": "Effective eddy-scale Reynolds number using U_eddy=sqrt(2*EKE), L_eddy, and nu_eff.",
        "MLD": "Mixed-layer depth from GLORYS or scalar user input.",
        "Ri_b": "Bulk Richardson number between surface and MLD depth using MITgcm T/S/u/v after spinup.",
    },

    "eke_mean_m2_s2": eke_mean,
    "eke_median_m2_s2": eke_median,
    "eke_std_m2_s2": eke_std,
    "u_eddy_m_s": u_eddy,

    "L_eddy_m": float(L_eddy_m),
    "nu_eff_m2_s": float(NU_EFF),
    "Re_eddy": re_eddy,

    "mld_mean_m": float(mld_stats["mld_mean_m"]),
    "mld_median_m": float(mld_stats["mld_median_m"]),
    "mld_std_m": None if not np.isfinite(mld_stats["mld_std_m"]) else float(mld_stats["mld_std_m"]),
    "mld_source": mld_stats["mld_source"],

    "Ri_b_median": rib_median,
    "Ri_b_mean": rib_mean,
    "Ri_b_p25": rib_p25,
    "Ri_b_p75": rib_p75,

    "rho0_kg_m3": RHO0,
    "alpha_T_1_K": ALPHA_T,
    "beta_S_1_psu": BETA_S,
    "g_m_s2": G,
}

with open(OUT_ATTRS_FILE, "w") as f:
    json.dump(attrs, f, indent=4)

print(f"\nSaved attrs file:\n{OUT_ATTRS_FILE}")

Opening MITgcm file:
C:\Users\Jelle Gortemaker\Documents\Thesis\z.flow_postprocessing\data\input\run_domain_characterization.nc


KeyError: "Could not find variable from candidates ['SALT', 'salt', 'salinity', 'S']. Available: ['diag_levels', 'iter', 'UVEL', 'VVEL', 'Xp1', 'Y', 'X', 'Yp1', 'T']"